# Kaggle run: measuring the execution barrier $P_{gen}$ (postscript)

This notebook reproduces the GPU measurement behind the report's Section
VI-G, "Measuring the execution barrier $P_{gen}$ (postscript)": a real,
joint `ASR_e2e(rho)` for the hybrid trigger/payload condition on a single
consumer GPU, plus the direct-feed probe that explains why it came out flat
zero. It runs the repo at
https://github.com/svmu92/rag-ipi-execution-barrier-postscript as-is —
`src/generate.py`'s generator loader already defaults to the AWQ/T4 settings
this needs, so unlike earlier drafts of this run, no in-notebook patching of
the repo's own files is required here. What *is* still needed is a handful
of Kaggle-environment-specific fixes below (HF token plumbing, a JAX/vLLM
fork quirk, a torchaudio import landmine) that have nothing to do with the
repo's own code and so live here instead.

## Before you start

- A Kaggle account with its phone number verified — Kaggle requires this
  before it will turn on GPU or internet access for a notebook.
- A Hugging Face account and access token (huggingface.co/settings/tokens,
  role **Read**). Gated access to `meta-llama/Meta-Llama-3-8B-Instruct`
  itself is **not** required for this run — the default generator is a
  public AWQ repack — but the token is still needed for Hugging Face
  downloads generally.
- Check the GPU quota shown in the notebook settings panel once you create
  the notebook (Kaggle gives roughly 30 GPU-hours per week); this run needs
  well under an hour, but confirm the week's quota isn't already spent.

## Step 1 — Create the notebook

Go to kaggle.com/code → **New Notebook**. In the settings panel on the
right:

- **Accelerator**: GPU T4 x2 (this run only actually uses one of the two
  T4s — see the generator-loading note in Step 4 below).
- **Internet**: On (needed for `git clone`, pip installs, and Hugging
  Face/dataset downloads).

## Step 2 — Clone the repo

In [ ]:
!git clone https://github.com/svmu92/rag-ipi-execution-barrier-postscript.git
%cd rag-ipi-execution-barrier-postscript

## Step 3 — Store your Hugging Face token, and fix two Kaggle/vLLM quirks

In the notebook's **Add-ons** menu, open **Secrets**, add a new secret named
`HF_TOKEN`, and paste your Hugging Face access token. Then:

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token  # some code paths look for this older name instead
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"  # see note below
login(token=hf_token)
print(whoami())

`HF_TOKEN` is set as an actual environment variable, not just via
`login()`'s cached token file — vLLM loads its model in a separate
subprocess of its own, and that subprocess reliably picks up an environment
variable but does not always find a token cached to disk by `login()`
alone. `whoami()` should print your account; if it errors or shows the
wrong one, the token is wrong or not set.

`VLLM_WORKER_MULTIPROC_METHOD=spawn` fixes a separate, unrelated problem:
Kaggle's notebook image has JAX preinstalled, and vLLM's GPU worker runs in
a subprocess it creates with Python's default `fork()` method on Linux —
forking after CUDA/JAX have touched the process breaks CUDA initialization
in that child (`RuntimeError: CUDA driver initialization failed`), even
though the GPU itself works fine. Forcing `spawn` avoids it. Set this once
per kernel session, before any script below that loads the generator. If
you restart the kernel, re-run this whole cell again.

There's a second, unrelated landmine here: transformers' own check for
whether torchaudio is installed only catches `ImportError`, but
torchaudio's CUDA-version guard raises `RuntimeError` when its build
doesn't match torch's CUDA version — that propagates straight through
`import transformers` itself. Since this project does zero audio
processing, the robust fix is to stub the module out:

In [ ]:
# Run once per kernel session, before Step 4's installs. If torchaudio
# already failed to import once this session, restart the kernel first.
!pip uninstall -y torchaudio -q

import sys, types
sys.modules["torchaudio"] = types.ModuleType("torchaudio")

## Step 4 — Install dependencies

Install `vllm` first so it pins its own compatible torch/CUDA build.

In [ ]:
!pip install vllm
!pip install -r requirements.txt

If `pip install vllm` errors out or `import vllm` fails afterward,
restart the kernel (**Run → Restart & Clear Output**) and try again, or add
`--no-build-isolation` to the vllm install.

## Step 5 — Run the environment smoke test

Run this directly in the notebook kernel, not as
`!python scripts/00_check_environment.py` — invoking it as a separate
subprocess was found to break vLLM's dtype resolution; running the
identical code in-process avoids it. `scripts/00_check_environment.py`'s
own defaults already point at the AWQ checkpoint and `dtype="half"` this
repo runs with (see `src/generate.py`'s `load_generator` docstring).

In [ ]:
import sys
sys.path.insert(0, "scripts")
sys.argv = ["00_check_environment.py"]
exec(open("scripts/00_check_environment.py").read())

If every check passes, you'll see
`An exception has occurred, use %tb to see the full traceback. SystemExit: 0`
right after `All checks passed.` — that's expected, not a crash: `main()`
calls `sys.exit(0)` on success, and since `exec()` runs this in the kernel
rather than as a real subprocess, Jupyter catches that `SystemExit` and
prints this notice instead of ending your session. Ignore it.

If Check 1 (vLLM generation) fails with an out-of-memory error: the
guardrail (Check 2, `device=0`) and vLLM (pinned to GPU 1 when two GPUs are
visible — see `load_generator`'s docstring) should not be contending for
the same GPU; if `torch.cuda.device_count()` is 1 in your session, lower
`gpu_memory_utilization` slightly instead.

## Step 6 — Download the corpus and sample queries

This is the slow step — it downloads the full BEIR Natural Questions corpus
and samples 30 eval queries from it (`configs/postscript.yaml`'s
`n_eval_queries`), seeded for reproducibility. Start it first and let it
run.

In [ ]:
!python scripts/01_prepare_data.py --config configs/postscript.yaml

## Step 7 — Run the real end-to-end pipeline

Run this directly in the kernel too, for the same reason as Step 5. This
loads the quantised model (a couple of minutes) before the sweep starts,
then runs real BM25 retrieval, real guardrail filtering and real generation
for the hybrid condition across all five $\rho$ values —
`scripts/06_end_to_end.py`, unmodified from the parent project.

In [ ]:
import sys
sys.path.insert(0, "scripts")
sys.argv = ["06_end_to_end.py", "--config", "configs/postscript.yaml"]
exec(open("scripts/06_end_to_end.py").read())

This produces a genuine `asr_measured` per $\rho$ for the hybrid
condition in `results/nq/end_to_end.jsonl` — not the report's derived
upper bound $ASR^{*} = P_{ret} \cdot \varepsilon$.

## Step 8 — Run the direct-feed $P_{gen}$ probe

If Step 7 comes out at or near zero across the board, this is the
follow-up that explains why: it bypasses retrieval and the guardrail
entirely and hands the adversarial passage to the generator as its sole
context, for 3 queries at $\rho \in \{0, 0.25, 0.5, 1\}$. $\rho=0$ is a
positive control (an unobfuscated instruction the model should comply with
readily, ruling out blanket refusal as the explanation).

In [ ]:
import sys
sys.path.insert(0, "scripts")
sys.argv = ["08_pgen_direct_feed_probe.py", "--config", "configs/postscript.yaml"]
exec(open("scripts/08_pgen_direct_feed_probe.py").read())

## Step 9 — Save and download your results

Files under `/kaggle/working` only exist for the life of the current
session. Two ways to get the results out safely:

- **While the session is still live**: open the file browser on the right
  (Output pane), find `results/nq/end_to_end.jsonl` and
  `results/nq/execution_postscript_probe.jsonl`, and use their
  **Download** option directly.
- **For a permanent copy**: click **Save Version** → **Save & Run All
  (Commit)**. This re-runs the whole notebook end-to-end and produces an
  **Output** tab on the saved version that stays downloadable even after
  the interactive session ends or your weekly quota resets.

## Reading the results

In [ ]:
import json

print("=== end_to_end.jsonl (real joint ASR_e2e) ===")
for line in open("results/nq/end_to_end.jsonl"):
    row = json.loads(line)
    print(f"condition={row['condition']}  rho={row['rho']:.2f}  defence={row['defence']}  "
          f"asr_measured={row['asr_measured']:.3f}")

print()
print("=== execution_postscript_probe.jsonl (direct-feed P_gen probe) ===")
from collections import defaultdict
by_rho = defaultdict(list)
for line in open("results/nq/execution_postscript_probe.jsonl"):
    row = json.loads(line)
    by_rho[row["rho"]].append(row["compliant"])
for rho, compliances in sorted(by_rho.items()):
    print(f"rho={rho:.2f}  compliance={sum(compliances)}/{len(compliances)}")

Compare `end_to_end.jsonl`'s `asr_measured` against the report's
hybrid-condition $ASR^{*}$ values (0.00, 0.00, 0.03, 0.09, 0.17 at
$\rho=0, 0.25, 0.5, 0.75, 1.0$) — since $ASR_{e2e} = P_{ret} \cdot
\varepsilon \cdot P_{gen}$ and $P_{gen} \le 1$, measured numbers should
sit at or below the matching $ASR^{*}$ at each $\rho$. The probe's
compliance counts are what explain *how far* below.

## Troubleshooting

| Symptom | Likely cause | Fix |
| --- | --- | --- |
| `403` loading a gated Hugging Face model | Gated access not yet approved for the account behind your token | Not expected with the default AWQ checkpoint (public); only relevant if you override `--generator` to the full-precision gated model |
| `401` "gated repo"/"please log in" right after a kernel restart | A kernel restart resets the in-memory login session | Re-run Step 3's login cell every time you restart the kernel, before anything downstream |
| CUDA out-of-memory despite the AWQ default | `max_model_len=4096` leaves too little headroom, or only one GPU is visible this session | Try lowering `max_model_len` in `src/generate.py`'s `load_generator` call to 2048, or `gpu_memory_utilization` to 0.75 |
| `pip install vllm` fails, or `import vllm` errors afterward | Version conflict with Kaggle's preinstalled torch | Restart the kernel after installing, or retry with `pip install vllm --no-build-isolation` |
| `CUDA driver initialization failed, you might not have a CUDA gpu` | vLLM's GPU worker subprocess uses `fork()` by default; JAX (preinstalled on Kaggle) touching the process before the fork breaks CUDA init in the child | Set `VLLM_WORKER_MULTIPROC_METHOD=spawn` before running any script that loads the generator (Step 3); re-run Step 3 again after any kernel restart |
| `transformers` import fails with a torchaudio `RuntimeError`/`ModuleNotFoundError` cascade | torchaudio's CUDA-version guard raises `RuntimeError`, which `transformers`' own `ImportError`-only check doesn't catch | Stub the module out (Step 3's second cell) before any other import that session; restart the kernel first if torchaudio already failed to import once |
| `ModuleNotFoundError: No module named 'common'` when running a step via `exec()` | `!python` automatically adds the script's own directory to `sys.path`; `exec()` inside the kernel doesn't | `sys.path.insert(0, "scripts")` before the `exec()` call (already in every code cell above) |
| `ValueError: File data/nq/corpus.jsonl not present!` even though `data/nq.zip` downloaded fully | `beir`'s `download_and_unzip` can silently fail to extract this archive even though the download succeeded | Extract manually: `!unzip -o data/nq.zip -d data/`, confirm with `!ls -la data/nq/`, then re-run Step 6 — it skips re-downloading once the files are already there |
| GPU quota exhausted mid-session | Kaggle's ~30 weekly GPU-hours used up | Wait for the weekly reset, or finish the run on a cheap GPU rental (RunPod/Vast.ai) instead |
| Session disconnects partway through | Browser closed, long idle gap, or a Kaggle infrastructure hiccup | Commit a Version (Step 9) as soon as results exist rather than leaving everything until the very end |
